In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import wandb
import os

# Set random seed for reproducibility
torch.manual_seed(42)

# Load and preprocess the data
class TemperatureDataset(Dataset):
    def __init__(self, sequence_length=10):
        # Load temperature dataset
        df = pd.read_csv('https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv')
        
        # Convert 'Date' to datetime and set as index
        df['Date'] = pd.to_datetime(df['Date'])
        df.set_index('Date', inplace=True)
        
        # Normalize the data
        self.scaler = MinMaxScaler()
        data = self.scaler.fit_transform(df[['Temp']].values)
        
        # Create sequences
        self.sequences = []
        self.targets = []
        
        for i in range(len(data) - sequence_length):
            self.sequences.append(data[i:i+sequence_length])
            self.targets.append(data[i+sequence_length])
            
        self.sequences = torch.FloatTensor(self.sequences)
        self.targets = torch.FloatTensor(self.targets)
        
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        return self.sequences[idx], self.targets[idx].squeeze()

# Define the LSTM model
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
        
    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        
        out, _ = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])
        return out.squeeze()

# Training function
def train_model(model, train_loader, criterion, optimizer, device, epoch):
    model.train()
    total_loss = 0
    
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        if batch_idx % 10 == 0:
            wandb.log({
                "batch_loss": loss.item(),
                "epoch": epoch,
                "batch": batch_idx
            })
    
    return total_loss / len(train_loader)

def main():
    os.environ["WANDB_API_KEY"] = "265429f30a001efd9e8ebb1a3b986c59b0ed621b"
    wandb.login()
    wandb.init(
        project="icid",
        name="LSTM-DDP",
        config={
            "learning_rate": 1e-3,
            "architecture": "LSTM",
            "dataset": "Windowed Time Series(2004-2017)",
            "epochs": 20,
            "batch_size": 64,
            "hidden_size": 64,
            "num_layers": 2
        }
    )

    
    # Set device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Create dataset and dataloader
    dataset = TemperatureDataset()
    train_size = int(0.8 * len(dataset))
    test_size = len(dataset) - train_size
    train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])
    
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
    
    # Initialize model
    model = LSTMModel(
        input_size=1,  # Single feature (temperature)
        hidden_size=64,
        num_layers=2
    ).to(device)
    
    # Define loss function and optimizer
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    
    # Training loop
    for epoch in range(20):
        train_loss = train_model(model, train_loader, criterion, optimizer, device, epoch)
        
        # Log metrics
        wandb.log({
            "epoch": epoch,
            "train_loss": train_loss
        })
        
        print(f"Epoch {epoch+1}/20, Loss: {train_loss:.4f}")
    
    # Close wandb run
    wandb.finish()

if __name__ == "__main__":
    main()

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


/tmp/ipykernel_8501/3544752387.py:36: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /opt/conda/conda-bld/pytorch_1708025847130/work/torch/csrc/utils/tensor_new.cpp:275.)
  self.sequences = torch.FloatTensor(self.sequences)


Epoch 1/20, Loss: 0.0596
Epoch 2/20, Loss: 0.0180
Epoch 3/20, Loss: 0.0123
Epoch 4/20, Loss: 0.0118
Epoch 5/20, Loss: 0.0115
Epoch 6/20, Loss: 0.0115
Epoch 7/20, Loss: 0.0119
Epoch 8/20, Loss: 0.0114
Epoch 9/20, Loss: 0.0111
Epoch 10/20, Loss: 0.0110
Epoch 11/20, Loss: 0.0109
Epoch 12/20, Loss: 0.0109
Epoch 13/20, Loss: 0.0105
Epoch 14/20, Loss: 0.0106
Epoch 15/20, Loss: 0.0103
Epoch 16/20, Loss: 0.0100
Epoch 17/20, Loss: 0.0095
Epoch 18/20, Loss: 0.0090
Epoch 19/20, Loss: 0.0090
Epoch 20/20, Loss: 0.0089


batch,▅▆▅▃▅█▁▃▆█▆▆█▅▃▆▆█▁▃▆█▁▃▆▁▃▅▆▁▅▆▁▃▆▃▁▃█▆
batch_loss,██▅▅▆▂▂▃▂▄▂▄▂▁▂▄▄▄▂▂▄▂▃▃▃▂▄▃▂▃▃▂▂▂▁▁▂▁▁▁
epoch,▁▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▇▇▇██
train_loss,█▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
batch,40
batch_loss,0.01243
epoch,19
train_loss,0.00891


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import wandb
import os
import matplotlib.pyplot as plt

# Set random seed for reproducibility
torch.manual_seed(42)

class TemperatureDataset(Dataset):
    def __init__(self, sequence_length=10):
        # Load temperature dataset
        df = pd.read_csv('https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv')
        
        # Convert 'Date' to datetime and set as index
        df['Date'] = pd.to_datetime(df['Date'])
        df.set_index('Date', inplace=True)
        
        # Normalize the data
        self.scaler = MinMaxScaler()
        self.data = self.scaler.fit_transform(df[['Temp']].values)
        
        # Create sequences
        self.sequences = []
        self.targets = []
        
        for i in range(len(self.data) - sequence_length):
            self.sequences.append(self.data[i:i+sequence_length])
            self.targets.append(self.data[i+sequence_length])
            
        self.sequences = torch.FloatTensor(self.sequences)
        self.targets = torch.FloatTensor(self.targets)
    
    def inverse_transform(self, data):
        return self.scaler.inverse_transform(data.reshape(-1, 1))
        
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        return self.sequences[idx], self.targets[idx].squeeze()

class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
        
    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        
        out, _ = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])
        return out.squeeze()

def train_model(model, train_loader, criterion, optimizer, device, epoch):
    model.train()
    total_loss = 0
    predictions = []
    actuals = []
    
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        predictions.extend(output.detach().cpu().numpy())
        actuals.extend(target.cpu().numpy())
        
        if batch_idx % 10 == 0:
            wandb.log({
                "batch_loss": loss.item(),
                "epoch": epoch,
                "batch": batch_idx
            })
    
    return total_loss / len(train_loader), predictions, actuals

def evaluate_model(model, data_loader, criterion, device, dataset, prefix='val'):
    model.eval()
    total_loss = 0
    predictions = []
    actuals = []
    
    with torch.no_grad():
        for data, target in data_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output, target)
            total_loss += loss.item()
            predictions.extend(output.cpu().numpy())
            actuals.extend(target.cpu().numpy())
    
    # Convert to original scale
    predictions = dataset.inverse_transform(np.array(predictions))
    actuals = dataset.inverse_transform(np.array(actuals))
    
    # Create prediction vs actual plot
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(predictions[:100], label='Predictions')
    ax.plot(actuals[:100], label='Actuals')
    ax.set_title(f'{prefix.capitalize()} Set: Predictions vs Actuals')
    ax.set_xlabel('Time Steps')
    ax.set_ylabel('Temperature')
    ax.legend()
    
    # Log to wandb
    wandb.log({
        f"{prefix}_loss": total_loss / len(data_loader),
        f"{prefix}_predictions_plot": wandb.Image(fig)
    })
    
    plt.close(fig)
    
    return total_loss / len(data_loader)

def main():
    os.environ["WANDB_API_KEY"] = "265429f30a001efd9e8ebb1a3b986c59b0ed621b"
    wandb.login()
    wandb.init(
        project="icid",
        name="LSTM-DDP",
        config={
            "learning_rate": 1e-3,
            "architecture": "LSTM",
            "dataset": "Windowed Time Series(2004-2017)",
            "epochs": 20,
            "batch_size": 64,
            "hidden_size": 64,
            "num_layers": 2
        }
    )
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    dataset = TemperatureDataset()
    train_size = int(0.7 * len(dataset))
    val_size = int(0.15 * len(dataset))
    test_size = len(dataset) - train_size - val_size
    
    train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(
        dataset, [train_size, val_size, test_size]
    )
    
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
    
    model = LSTMModel(
        input_size=1,
        hidden_size=64,
        num_layers=2
    ).to(device)
    
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    
    best_val_loss = float('inf')
    for epoch in range(20):
        train_loss, train_preds, train_actuals = train_model(
            model, train_loader, criterion, optimizer, device, epoch
        )
        
        val_loss = evaluate_model(model, val_loader, criterion, device, dataset, 'val')
        
        wandb.log({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss
        })
        
        print(f"Epoch {epoch+1}/20, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), 'best_model.pth')
    
    # Load best model and evaluate on test set
    model.load_state_dict(torch.load('best_model.pth'))
    test_loss = evaluate_model(model, test_loader, criterion, device, dataset, 'test')
    print(f"Test Loss: {test_loss:.4f}")
    
    wandb.finish()

if __name__ == "__main__":
    main()

In [1]:
import os
import csv
import time
import wandb
import math
import numpy as np
import dask.dataframe as dd

import torch
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_curve, confusion_matrix, auc
from sklearn.model_selection import train_test_split

from dataset import WindowedDataset
from model import LSTMClassifier
data_dir = "/root/data/rrr/integrated_weather_dataset/data/integrated/parquet_fixed"


In [3]:
parquet_file = f"{data_dir}/2006.parquet",
    
df = dd.read_parquet(parquet_file, blocksize='32MB')  
df['Label'] = (df['Guan_Label_approx'].astype(int) | df['Rutz_Label_approx'].astype(int))
df = df.drop(columns=["Guan_Label_approx", "Rutz_Label_approx"])
df = df.sort_values(by=['Site', 'Timestamp'])
df = df.reset_index()
df = df.drop(columns=["index"])
df = df.rename(columns={'level_0': 'index'})
df['Timestamp'] = df['Timestamp'].round('5T')
df = df.compute()